# Day 19: Advanced Indexing - MultiIndex

## Creating a MultiIndex
Real-world data is often naturally hierarchical — sales by region and then by product, or measurements by year and then by month. A MultiIndex lets a DataFrame's rows be identified by a combination of keys instead of just one, which mirrors that structure directly instead of forcing you to flatten it into a single column. This becomes especially useful once you start grouping or aggregating, since operations can act on just one level of the hierarchy at a time (e.g. "sum by Region" or "sum by Product"). Below, `set_index(['Region', 'Product'])` turns those two columns into a two-level row index — printing `df` shows the data with Region and Product stacked as row labels instead of regular columns, and Sales as the only remaining column.</cell id="cell-1">


In [ ]:
import pandas as pd

data = {
    'Region': ['East', 'East', 'West', 'West'],
    'Product': ['A', 'B', 'A', 'B'],
    'Sales': [100, 150, 200, 130]
}
# set_index with a list of columns creates a two-level (hierarchical) MultiIndex from Region + Product
df = pd.DataFrame(data).set_index(['Region', 'Product'])
print(df)

## Selecting from a MultiIndex
Once a DataFrame has a MultiIndex, you need different ways to slice into it depending on how specific you want to be. `.loc['East']` selects everything under the outer (first) level, `.loc[('East', 'A')]` uses a tuple to pin down an exact combination of both levels, and `.xs('A', level='Product')` ("cross-section") selects by a value from an *inner* level while ignoring the outer one — something plain `.loc` tuples can't easily do. These patterns come up constantly in DS work when you've grouped data hierarchically and need to pull out one region, one exact region/product pair, or every row for a given product across all regions. Read the three printed outputs as three different "views" of the same underlying table, each narrowing the selection a different way.</cell id="cell-3">


In [ ]:
print(df.loc['East'])  # all rows where the outer index level (Region) is 'East'
print(df.loc[('East', 'A')])  # the exact row matching Region='East' AND Product='A'
print(df.xs('A', level='Product'))  # all rows where Product='A', across every Region

## Resetting the Index
A MultiIndex is powerful for selection but not always convenient — for exporting to CSV, merging with another flat DataFrame, or feeding into a plotting/ML library, you often want plain columns instead of index levels. `reset_index()` undoes the hierarchy by moving every index level back into a regular column and replacing the row index with default integers. This is a common "flatten it back out" step after you're done doing level-based operations. The printed output shows Region and Product as normal columns again, alongside Sales, with a fresh 0, 1, 2, 3 row index.</cell id="cell-5">


In [ ]:
flat = df.reset_index()  # move index levels (Region, Product) back into regular columns
print(flat)

## Unstack & Stack
`unstack` and `stack` reshape data between "long" format (many rows, few columns — good for storage/tidy data) and "wide" format (fewer rows, more columns — good for readability and comparison, like a pivot table). `unstack` takes an index level and pivots its unique values into columns, while `stack` does the reverse, folding columns back down into an index level. This is one of the most common reshaping operations in data analysis — for example, turning a "one row per Region-Product" table into a "one row per Region, one column per Product" summary table that's easier to eyeball. In the output, `wide` shows Sales broken into separate columns for Product A and B (with Region as rows), and `wide.stack()` converts it back to the original long shape.</cell id="cell-7">


In [ ]:
# pivot the 'Product' index level into columns -> wide format (Region as rows, Product as columns)
wide = df.unstack('Product')
print(wide)
print(wide.stack())  # fold the Product columns back into the index -> back to long format